# Smart Parking System — Evaluation

This notebook produces the numbers for your report/slides. It has two parts:

1. **Model detection metrics** — how well `best.pt` detects parking spaces (mAP, precision, recall). Uses the GPU, so set **Runtime → Change runtime type → GPU**.
2. **End-to-end occupancy metrics** — how well the *whole system* decides occupied/empty, compared to a ground truth you provide (accuracy, precision, recall, F1, confusion matrix).

Part 2 works directly from the `parking_history.csv` your `main.py` already produces.

In [ ]:
# Install dependencies (most are preinstalled on Colab)
!pip -q install ultralytics scikit-learn seaborn pandas matplotlib

## Part 1 — Model detection metrics (`best.pt`)

Upload `best.pt` and your dataset config `data.yaml` (the one you trained with, pointing at the validation images + labels). Easiest is to mount Google Drive.

If you already trained the model, YOLO already computed these numbers — they're in `runs/detect/train/results.csv`. Re-running `model.val()` here just regenerates them cleanly.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# --- Option A: mount Google Drive (recommended) ---
# from google.colab import drive
# drive.mount('/content/drive')
# MODEL_PATH = '/content/drive/MyDrive/smart_parking/best.pt'
# DATA_YAML  = '/content/drive/MyDrive/smart_parking/data.yaml'

# --- Option B: upload directly ---
# from google.colab import files
# files.upload()   # pick best.pt and data.yaml

MODEL_PATH = 'best.pt'      # <-- path to your trained model
DATA_YAML  = 'data.yaml'    # <-- dataset config with the validation set

if Path(MODEL_PATH).exists() and Path(DATA_YAML).exists():
    model = YOLO(MODEL_PATH)
    metrics = model.val(data=DATA_YAML)
    print('mAP@50     :', round(float(metrics.box.map50), 4))
    print('mAP@50-95  :', round(float(metrics.box.map), 4))
    print('Precision  :', round(float(metrics.box.mp), 4))
    print('Recall     :', round(float(metrics.box.mr), 4))
else:
    print('Skipped: best.pt or data.yaml not found here.')
    print('Upload them, or just reuse the precision/recall/mAP from your training')
    print('output at runs/detect/train/results.csv.')

In [ ]:
# Show the plots YOLO saved (confusion matrix, PR curve, training curves) if present
import glob
from IPython.display import Image, display
found = False
for pat in ['runs/detect/val*/confusion_matrix.png',
            'runs/detect/val*/PR_curve.png',
            'runs/detect/train*/results.png',
            'runs/detect/train*/confusion_matrix.png']:
    for p in sorted(glob.glob(pat)):
        found = True
        print(p); display(Image(filename=p))
if not found:
    print('No saved plots found yet (run the cell above, or upload your runs/ folder).')

## Part 2 — End-to-end occupancy metrics

**Two files are needed:**

- `parking_history.csv` — produced by `main.py` (columns: `Event, Spot_ID, Vehicle_ID, Video_Timestamp, Real_Time, Image_Path`).
- `ground_truth.csv` — **you create this.** Scrub the test video at a fixed interval and record the true state of each spot:

| time | spot_id | occupied |
|------|---------|----------|
| 00:00 | 1 | 0 |
| 00:00 | 2 | 1 |
| 00:05 | 1 | 1 |
| ...  | ... | ... |

`time` is `MM:SS` (or seconds), `spot_id` matches the `P` number, `occupied` is `1` or `0`.

Run the optional template cell below to generate a blank `ground_truth.csv` to fill in.

In [ ]:
import pandas as pd

# Upload both files, or set paths if mounted from Drive
# from google.colab import files
# files.upload()   # choose parking_history.csv (and ground_truth.csv once filled)

EVENTS_CSV = 'parking_history.csv'   # system output from main.py
TRUTH_CSV  = 'ground_truth.csv'      # the file you fill in

In [ ]:
# Load the event log and build a function that gives the system's
# predicted state of a spot at any time = the last ENTRY/EXIT before that time.
def to_seconds(t):
    if isinstance(t, (int, float)):
        return float(t)
    t = str(t).strip()
    if ':' in t:
        m, s = t.split(':')
        return int(m) * 60 + int(s)
    return float(t)

events = pd.read_csv(EVENTS_CSV)
events['t'] = events['Video_Timestamp'].map(to_seconds)
events = events.sort_values('t')
NUM_SPOTS = int(events['Spot_ID'].max()) if len(events) else 0
print(f'Loaded {len(events)} events across {NUM_SPOTS} spots.')

def predicted_occupied(spot_id, t):
    e = events[(events['Spot_ID'] == spot_id) & (events['t'] <= t)]
    if e.empty:
        return 0
    return 1 if str(e.iloc[-1]['Event']).upper() == 'ENTRY' else 0

In [ ]:
# OPTIONAL: make a blank ground-truth template to fill in.
SAMPLE_EVERY_SEC = 5
VIDEO_LENGTH_SEC = 120          # <-- set your clip length in seconds
SPOTS = NUM_SPOTS if NUM_SPOTS else 4

rows = []
for t in range(0, VIDEO_LENGTH_SEC + 1, SAMPLE_EVERY_SEC):
    for s in range(1, SPOTS + 1):
        rows.append({'time': f'{t//60:02d}:{t%60:02d}', 'spot_id': s, 'occupied': ''})
pd.DataFrame(rows).to_csv('ground_truth_TEMPLATE.csv', index=False)
print('Wrote ground_truth_TEMPLATE.csv')
print('Fill the "occupied" column with 1/0, save it as ground_truth.csv, and upload.')

In [ ]:
# Align ground truth with predictions and compute metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

truth = pd.read_csv(TRUTH_CSV)
truth['t'] = truth['time'].map(to_seconds)
truth = truth.dropna(subset=['occupied'])
truth['occupied'] = truth['occupied'].astype(int)

y_true, y_pred, spot_ids = [], [], []
for _, r in truth.iterrows():
    y_true.append(int(r['occupied']))
    y_pred.append(predicted_occupied(int(r['spot_id']), r['t']))
    spot_ids.append(int(r['spot_id']))

print('Samples    :', len(y_true))
print('Accuracy   :', round(accuracy_score(y_true, y_pred), 4))
print('Precision  :', round(precision_score(y_true, y_pred, zero_division=0), 4))
print('Recall     :', round(recall_score(y_true, y_pred, zero_division=0), 4))
print('F1-score   :', round(f1_score(y_true, y_pred, zero_division=0), 4))
print()
print(classification_report(y_true, y_pred, target_names=['Empty', 'Occupied'], zero_division=0))

In [ ]:
# Confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Empty', 'Occupied'], yticklabels=['Empty', 'Occupied'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Occupancy Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# Per-spot accuracy
df = pd.DataFrame({'spot': spot_ids, 'true': y_true, 'pred': y_pred})
df['correct'] = (df['true'] == df['pred']).astype(int)
per_spot = df.groupby('spot')['correct'].mean().reset_index()
per_spot.columns = ['Spot', 'Accuracy']
per_spot['Accuracy'] = (per_spot['Accuracy'] * 100).round(1)
print(per_spot.to_string(index=False))

plt.figure(figsize=(6, 3.5))
plt.bar(per_spot['Spot'].astype(str), per_spot['Accuracy'], color='#4C78A8')
plt.ylim(0, 100); plt.ylabel('Accuracy (%)'); plt.xlabel('Spot')
plt.title('Per-spot occupancy accuracy'); plt.tight_layout(); plt.show()

In [ ]:
# Occupancy over time: actual vs predicted
times = sorted(truth['t'].unique())
true_counts = [int(truth[truth['t'] == t]['occupied'].sum()) for t in times]
pred_counts = [sum(predicted_occupied(s, t) for s in range(1, NUM_SPOTS + 1)) for t in times]
plt.figure(figsize=(8, 3.5))
plt.plot([t/60 for t in times], true_counts, label='Actual', marker='o')
plt.plot([t/60 for t in times], pred_counts, label='Predicted', marker='x')
plt.xlabel('Time (min)'); plt.ylabel('Occupied spots'); plt.legend()
plt.title('Occupancy over time: actual vs predicted'); plt.tight_layout(); plt.show()

## Putting it in the report

- **Part 1** gives the *model* quality: `mAP@50`, precision, recall — cite these for `best.pt`.
- **Part 2** gives the *system* quality: accuracy, precision, recall, F1 + the confusion matrix and per-spot table.

Drop the confusion matrix and the per-spot bar chart straight into the slides — they read well visually and back up the live demo with hard numbers.